> ### Fresh run (Cursor)
>
> 1. `Cmd + Shift + P`
> 2. **`Jupyter: Restart Kernel and Run All Cells`**
> 3. Confirm if asked

# PigeonPilot

**Research question:** Does STDP in a recurrent LIF reservoir improve path integration compared to a fixed reservoir?

Logic lives in `pigeonpilot/`. This notebook only explains and runs it.

1. Displacement paths + curriculum  
2. Spike encoding — *next*  
3. Reservoir A vs B — *next*  
4. Readout — *next*

## 1. Displacement environment

Home = `(0, 0)`. A **level** = segments `(heading, distance)`.

### Two naming layers (do not mix them up)

**A) Trajectory family** = *shape* of the path

| Name | Meaning |
|------|--------|
| `linear` | straight / direct |
| `turning` | heading jumps (turns) |
| `curved` | wave or sweeping arc |

**B) Difficulty tag** = *curriculum stage*

| Tag | Meaning |
|-----|--------|
| `easy` / `medium` / `hard` | how hard path integration is |

- **Displacement vector:** home → release  
- **Home vector:** release → home

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from pigeonpilot.paths import (
    STYLES,
    generate_level,
    generate_curriculum_dataset,
    split_dataset,
    summarize_dataset,
    iter_train_schedule,
)
from pigeonpilot.viz import (
    plot_level,
    plot_levels_grid,
    plot_release_points,
    compute_xy_limits,
)

print("imports ok")
print("trajectory families:", STYLES)
print("cwd:", ROOT)

### 1.1 One example per trajectory family

`linear` / `turning` / `curved` — shared x/y scale.

In [ ]:
demo = [
    generate_level(style=s, n_segments=3, seed=0, level_id=i, turning_scale="gentle")
    for i, s in enumerate(("linear", "turning", "curved"))
]
xlim, ylim = compute_xy_limits(demo)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, level in zip(axes, demo):
    plot_level(level, ax=ax, xlim=xlim, ylim=ylim)
plt.tight_layout()
plt.show()

### 1.2 Curriculum: difficulty tags × trajectory families

| Difficulty | Count | Families used | Segments | Skill |
|------------|-------|---------------|----------|-------|
| **easy** | 60 | `linear` + gentle `turning` | 2–3, short | distance + mild turn |
| **medium** | 60 | sharp `turning` + mild `curved` | 4–5 | many turns/curves (**no linear**) |
| **hard** | 30 | sharp `turning` + **arc** `curved` | 6–8, longer | long memory + circle-like arcs |

**Total = 150 levels.** Then ~80% train / ~20% test.

### 1.3 Generate all 150 levels and plot them

Must print `total: 150`.

One figure per difficulty. Each grid zooms to that group only (square axes around home).


In [ ]:
DATASET_SEED = 42
SPLIT_SEED = 0

dataset = generate_curriculum_dataset(seed=DATASET_SEED)
print(summarize_dataset(dataset))
assert len(dataset) == 150, f"expected 150, got {len(dataset)}"

train_set, test_set = split_dataset(dataset, train_frac=0.8, seed=SPLIT_SEED)
print(f"train: {len(train_set)} | test: {len(test_set)}")

# Global window only for the release map in 1.5
xlim_global, ylim_global = compute_xy_limits(dataset)

for difficulty in ("easy", "medium", "hard"):
    subset = [lv for lv in dataset if lv.difficulty == difficulty]
    plot_levels_grid(subset, title=f"Difficulty: {difficulty}")
    plt.show()


### 1.4 Training order

`SCHEDULE_MODE`:

- `"curriculum"` → **easy → medium → hard** (shuffle inside each difficulty)
- `"mixed"` → shuffle all train levels every epoch

Only `train_set` is used.

In [ ]:
SCHEDULE_MODE = "curriculum"  # or "mixed"
EPOCHS_PER_DIFFICULTY = {"easy": 40, "medium": 30, "hard": 30}

schedule = list(
    iter_train_schedule(
        train_set,
        mode=SCHEDULE_MODE,
        epochs_per_difficulty=EPOCHS_PER_DIFFICULTY,
        seed=0,
    )
)

from collections import Counter

print("mode:", SCHEDULE_MODE)
print("presentations:", len(schedule))
print("per phase:", dict(Counter(phase for phase, *_ in schedule)))
print("first:", schedule[0][0], schedule[0][3].style, schedule[0][3].difficulty)
print("last:", schedule[-1][0], schedule[-1][3].style, schedule[-1][3].difficulty)

### 1.5 Release-point map

Each dot = path end (release). Star = home. Color = difficulty.
Uses one global window so absolute distances stay comparable.


In [ ]:
for level in dataset:
    assert np.allclose(level.home_xy, (-level.end_xy[0], -level.end_xy[1]))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_release_points(
    train_set, ax=axes[0], title="Train release points",
    xlim=xlim_global, ylim=ylim_global,
)
plot_release_points(
    test_set, ax=axes[1], title="Test release points",
    xlim=xlim_global, ylim=ylim_global,
)
plt.tight_layout()
plt.show()

print(f"section 1 done | {len(dataset)} levels | train {len(train_set)} | test {len(test_set)}")


---

## 2. Spike encoding *(coming next)*

## 3. Reservoir A vs B *(coming next)*

## 4. Readout *(coming next)*